In [1]:
# === SESSION BOOTSTRAP ===
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, sys
PARENT="/content/drive/MyDrive/UAV_TRUST_Research"; REPO=f"{PARENT}/uav-trust-research"
for fn in (".gitconfig",".git-credentials"):
    p=os.path.join(PARENT,fn)
    if os.path.exists(p): subprocess.run(f'cp "{p}" /root/{fn}',shell=True)
subprocess.run("git config --global credential.helper store",shell=True)
if os.path.isdir(REPO):
    os.chdir(REPO); sys.path.insert(0,REPO) if REPO not in sys.path else None; print("cwd:",os.getcwd())
else: print("run 00_setup first")

Mounted at /content/drive
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost scikit-learn pandas numpy requests --quiet

In [3]:
# ATTACK-CONDITIONAL prediction-set decomposition: compute empty / {normal} / {attack} / {both}
# ONLY on the held-out ATTACK records, so safe failure (empty=abstention) is separated from
# dangerous failure ({normal}=confident wrong) correctly, and likewise on NORMAL records separately.
DATASETS=[
 {"name":"UAVIDS-2025","kind":"zenodo","record":"15336998","data_dir":"data/uavids2025",
  "label_col":"label","normal_value":"Normal Traffic","include_families":None,"subsample_n":None,
  "drops":["unnamed","flowid","srcaddr","dstaddr","srcport","dstport","index","timestamp"]},
 {"name":"UAV-NIDD","kind":"file","file":"data/uav_nidd/UAV-NDD CSV/UAV-Case1-Label.csv","parquet":"data/uav_nidd/case1.parquet",
  "data_dir":"data/uav_nidd","label_col":"Label","normal_value":"Normal",
  "include_families":["DDoS","UDP Flooding","MITM","Jamming","BruteForce","De-authentication"],"subsample_n":200000,
  "drops":["unnamed","index","ip.src","ip.dst","ip.proto","wlan.tag","srcport","dstport","udp.srcport","udp.dstport",
           "frame.time","frame.number","time_epoch","time_relative","time_delta","bssid","mactime",
           "vendor_oui","wlan_radio.timestamp","wlan_radio.start_tsf","radiotap.timestamp","wlan.seq"]},
 {"name":"UAV-CAS","kind":"cas","file":"data/uav_cas/UAV-CAS_stat.csv","label_col":"Label","normal_value":"Benign",
  "single":{"benign","dos","ddos","blackhole","wormhole","replay"},
  "drops":["config_idx","num_drones","num_bs","payload","pathloss","modulation","mission","tx_power","noise","src_ip","dst_ip"]},
]
CFG={"seeds":list(range(10)),"alpha":0.10,
     "normal_fracs":{"train":0.60,"cal":0.20,"test_seen":0.10,"test_shift":0.10},
     "family_fracs":{"train":0.60,"cal":0.20,"test_seen":0.20},
     "xgb":{"n_estimators":300,"max_depth":6,"learning_rate":0.1,"subsample":0.9,"colsample_bytree":0.9,"tree_method":"hist"},
     "report_dir":"reports"}
import os
os.makedirs(CFG["report_dir"],exist_ok=True); print("configured")

configured


In [4]:
import numpy as np, pandas as pd, requests, glob, zipfile, importlib, gc, src.data
importlib.reload(src.data)
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from src.data import load_csvs, detect_schema, prepare_splits
from src.trust import conformal_qhat

def load_any(spec):
    if spec["kind"]=="zenodo":
        dd=spec["data_dir"]; os.makedirs(dd,exist_ok=True)
        if not glob.glob(dd+"/**/*.csv",recursive=True):
            meta=requests.get(f"https://zenodo.org/api/records/{spec['record']}",timeout=60).json()
            for f in meta.get("files",[]):
                n,u=f["key"],f["links"]["self"]
                if n.lower().endswith((".csv",".zip",".gz")): open(os.path.join(dd,n),"wb").write(requests.get(u,timeout=1200).content)
            for z in glob.glob(dd+"/*.zip"): zipfile.ZipFile(z).extractall(dd)
        df=load_csvs(dd); lc,nv,fams=detect_schema(df,spec["label_col"],spec["normal_value"]); return df,lc,nv,fams,"pipeline"
    if spec["kind"]=="file":
        pq=spec.get("parquet"); df=pd.read_parquet(pq) if pq and os.path.exists(pq) else pd.read_csv(spec["file"],low_memory=False,encoding="latin-1")
        lc,nv=spec["label_col"],spec["normal_value"]; fams=list(spec["include_families"])
        if spec.get("subsample_n") and len(df)>spec["subsample_n"]: df=df.groupby(lc,group_keys=False).sample(frac=spec["subsample_n"]/len(df),random_state=42).reset_index(drop=True)
        df=df[df[lc].isin([nv]+fams)].reset_index(drop=True); return df,lc,nv,fams,"pipeline"
    # UAV-CAS
    raw=pd.read_csv(spec["file"],low_memory=False); lab=raw[spec["label_col"]].astype(str)
    keep=lab.str.lower().isin(spec["single"]) & (~lab.str.contains("+",regex=False)); df=raw[keep].reset_index(drop=True)
    nv=[v for v in df[spec["label_col"]].unique() if str(v).lower()==spec["normal_value"].lower()][0]
    fams=[v for v in df[spec["label_col"]].unique() if v!=nv]; return df,spec["label_col"],nv,fams,"cas"

def cas_prepare(df,lc,nv,fams,held_out,drops,seed):
    feat=df.drop(columns=[col for col in drops if col in df.columns]+[lc]).apply(pd.to_numeric,errors="coerce")
    feat=feat.drop(columns=[col for col in feat.columns if feat[col].nunique(dropna=False)<=1]).replace([np.inf,-np.inf],np.nan).fillna(0.0)
    X=feat.values.astype(np.float32); y=(df[lc].values!=nv).astype(int); fam=df[lc].values; seen=[f for f in fams if f!=held_out]
    def sp(ix,fr,sd):
        ix=np.array(ix); r=np.random.default_rng(sd); r.shuffle(ix); out={}; s=0; ks=list(fr)
        for i,k in enumerate(ks): e=len(ix) if i==len(ks)-1 else s+int(round(fr[k]*len(ix))); out[k]=ix[s:e]; s=e
        return out
    tr=[];ca=[];se=[];sh=[]
    ns=sp(np.where(fam==nv)[0],CFG["normal_fracs"],seed); tr+=list(ns["train"]);ca+=list(ns["cal"]);se+=list(ns["test_seen"]);sh+=list(ns["test_shift"])
    for j,f in enumerate(seen): fs=sp(np.where(fam==f)[0],CFG["family_fracs"],seed+j+1); tr+=list(fs["train"]);ca+=list(fs["cal"]);se+=list(fs["test_seen"])
    sh+=list(np.where(fam==held_out)[0]); tr,ca,se,sh=(np.array(sorted(a)) for a in (tr,ca,se,sh))
    scaler=StandardScaler().fit(X[tr]); tf=lambda ix: scaler.transform(X[ix]).astype(np.float32)
    return {"X_train":tf(tr),"y_train":y[tr],"X_cal":tf(ca),"y_cal":y[ca],"X_shift":tf(sh),"y_shift":y[sh],"fam_shift":fam[sh]}
print("loaders ready")

loaders ready


In [5]:
# For each dataset/family/seed: conformal, then decompose sets on ATTACK records and NORMAL records SEPARATELY
def decomp(p,mask,qh):
    p=np.asarray(p)[mask];
    if len(p)==0: return dict(n=0,empty=np.nan,normal=np.nan,attack=np.nan,both=np.nan)
    ina=(1-p)<=qh; ino=p<=qh
    return dict(n=int(len(p)),empty=float(((~ina)&(~ino)).mean()),
                normal=float((ino&(~ina)).mean()),attack=float((ina&(~ino)).mean()),both=float((ina&ino).mean()))
rows=[]
for spec in DATASETS:
    df,lc,nv,fams,mode=load_any(spec); print(spec["name"],"loaded",len(df))
    for F in fams:
        for seed in CFG["seeds"]:
            if mode=="cas": S=cas_prepare(df,lc,nv,fams,F,spec["drops"],seed)
            else:
                Sp=prepare_splits(df,lc,nv,F,spec["drops"],CFG["normal_fracs"],CFG["family_fracs"],seed)
                S={"X_train":Sp["X_train"].astype(np.float32),"y_train":Sp["y_train"],"X_cal":Sp["X_cal"].astype(np.float32),
                   "y_cal":Sp["y_cal"],"X_shift":Sp["X_shift"].astype(np.float32),"y_shift":Sp["y_shift"],"fam_shift":Sp["fam_shift"]}
            clf=xgb.XGBClassifier(objective="binary:logistic",eval_metric="logloss",random_state=seed,**CFG["xgb"]).fit(S["X_train"],S["y_train"])
            p_cal=clf.predict_proba(S["X_cal"])[:,1]; qh=conformal_qhat(p_cal,S["y_cal"],alpha=CFG["alpha"])
            p_sh=clf.predict_proba(S["X_shift"])[:,1]
            atk_mask=(S["fam_shift"]==F); norm_mask=(S["y_shift"]==0)
            da=decomp(p_sh,atk_mask,qh); dn=decomp(p_sh,norm_mask,qh)
            rows.append({"dataset":spec["name"],"held_out":str(F),"seed":seed,"one_minus_q":float(1-qh),
                "atk_empty":da["empty"],"atk_wrong_normal":da["normal"],"atk_correct_attack":da["attack"],"atk_both":da["both"],
                "norm_correct_normal":dn["normal"],"norm_wrong_attack":dn["attack"],"norm_empty":dn["empty"],"norm_both":dn["both"]})
            del S,clf; gc.collect()
        print(" ",spec["name"],F,"done")
    del df; gc.collect()
D=pd.DataFrame(rows); D.to_csv(os.path.join(CFG["report_dir"],"17_attack_conditional_decomp_raw.csv"),index=False)
print("rows:",len(D))

UAVIDS-2025 loaded 122171
  UAVIDS-2025 Sybil Attack done
  UAVIDS-2025 Blackhole Attack done
  UAVIDS-2025 Wormhole Attack done
  UAVIDS-2025 Flooding Attack done
UAV-NIDD loaded 168881
  UAV-NIDD DDoS done
  UAV-NIDD UDP Flooding done
  UAV-NIDD MITM done
  UAV-NIDD Jamming done
  UAV-NIDD BruteForce done
  UAV-NIDD De-authentication done
UAV-CAS loaded 92131
  UAV-CAS DoS done
  UAV-CAS DDoS done
  UAV-CAS Blackhole done
  UAV-CAS Wormhole done
  UAV-CAS Replay done
rows: 150


In [6]:
# HEADLINE: on the held-out ATTACK records -- safe failure (empty) vs dangerous failure (wrong {normal})
A=D.groupby(["dataset","held_out"]).agg(
    one_minus_q=("one_minus_q","mean"),
    safe_empty=("atk_empty","mean"), dangerous_wrong_normal=("atk_wrong_normal","mean"),
    correct_attack=("atk_correct_attack","mean"), uninformative_both=("atk_both","mean")).round(3)
print("=== ATTACK-conditional decomposition (held-out attack records only) ===")
print("safe_empty = abstention; dangerous_wrong_normal = confident WRONG (attack called normal)")
print(A.to_string())
A.to_csv(os.path.join(CFG["report_dir"],"17_attack_conditional_decomp.csv"))
print("\n=== NORMAL-record check (should be mostly correct {normal}) ===")
Nrm=D.groupby(["dataset","held_out"]).agg(correct_normal=("norm_correct_normal","mean"),wrong_attack=("norm_wrong_attack","mean")).round(3)
print(Nrm.to_string())

=== ATTACK-conditional decomposition (held-out attack records only) ===
safe_empty = abstention; dangerous_wrong_normal = confident WRONG (attack called normal)
                               one_minus_q  safe_empty  dangerous_wrong_normal  correct_attack  uninformative_both
dataset     held_out                                                                                              
UAV-CAS     Blackhole                0.197       0.000                   0.654           0.001               0.346
            DDoS                     0.265       0.000                   0.001           0.996               0.003
            DoS                      0.269       0.000                   0.000           1.000               0.000
            Replay                   0.258       0.000                   0.461           0.001               0.537
            Wormhole                 0.208       0.000                   0.700           0.001               0.298
UAV-NIDD    BruteForce            

In [ ]:
!git add reports/ notebooks/
!git commit -m "17 attack-conditional prediction-set decomposition: separate safe failure (empty) from dangerous failure (confident wrong normal) on held-out attack records, all three datasets"
!git push origin main